# Chapter 4. State Estimation — 상태 추정

Rawlings, Mayne, Diehl (2017), **2판 1쇄 제4장, 인쇄 pp.269–338**에 맞춘 한글 해설·실습입니다.
교재의 주요 절 4.1–4.8과 모든 하위 절 **18개**를 같은 순서로 배치했습니다.

**학습 목표**
- 측정 잔차와 과정 외란을 분리하여 전체 정보 추정(FIE)을 구성한다.
- 선형 가우스 FIE·칼만 필터(KF)·정확한 도착 비용의 MHE를 같은 시점에서 비교한다.
- 사전 정보, 관측가능성, 상태 제약과 평활화의 역할을 구분한다.
- 유계 잡음의 해석적 추정 범위와 확률적 추정 결과를 구분한다.
- 입자 필터·EKF·UKF·MHE의 표현과 계산 차이를 이해한다.
- MHE 추정값을 MPC에 연결하고 추정·제어의 결합 가정을 확인한다.

- RMSE와 NIS·NEES·구간 포함률로 추정 평균과 공분산을 따로 검증한다.

**시간 표기 — 중요**
교재 p.271 각주처럼 기본 실습은 $y_0,\ldots,y_{T-1}$로 $x_T$를 추정하는 **예측형**입니다.
$m_k^-=E[x_k\mid y_{0:k-1}]$, $m_k^+=E[x_k\mid y_{0:k}]$로 표기합니다.
비선형에서는 같은 기호가 해당 필터의 근사 평균을 뜻합니다. MHE의 해는 일반적으로 조건부 평균과 같지 않습니다.
§4.3.4는 평활화, §4.6은 현재 측정까지 반영한 필터링형을 별도로 명시합니다.

**실행:** 코랩에서 `런타임 → 모두 실행`. 전체 학습은 위에서 아래로 실행하세요. 개별 실습은 공통 준비 코드 두 셀 이후 독립적으로 실행할 수 있습니다.
이전 장 실행 결과나 저장소 복제가 필요하지 않습니다. CPU만 사용하며 최초 설치·글꼴 다운로드에는 인터넷이 필요합니다.

**범위:** 이론은 원문에 근거해 한국어로 재서술했습니다. Example 4.40의 충분조건과 Example 4.41의 명목 반응기 모델을 원본에 연결하고 나머지는 직접 설계한 **보충 실습**입니다.
교재의 모든 수치 사례·증명·연습문제 해답을 재현한 자료는 아닙니다. 문헌과 안정성 결과는 해당 2017년 판의 가정을 기준으로 설명합니다.

**기호:** 추정 잡음 공분산은 $Q_w,R_v$, 사전 공분산은 $P$, 최적화 상태 변수는 $\xi$입니다.
제어 비용 가중치는 $Q_c,R_c$로 구분합니다. 보충 최적화 비용은 공통 계수 1/2를 생략합니다. Example 4.40의 분석 상수는 원본의 1/2를 유지합니다.

[PPT](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter04_state_estimation/slides/chapter04_slides.pptx)
· [PDF](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter04_state_estimation/slides/chapter04_slides.pdf)

## 실습 환경 준비

필요한 패키지·글꼴과 공통 함수를 준비합니다. 솔버 상태와 수치 오차를 검사하며 실패를 숨기지 않습니다.

In [ ]:
import importlib.util, subprocess, sys

packages = {
    "numpy": "numpy>=2,<3",
    "scipy": "scipy>=1.14,<2",
    "matplotlib": "matplotlib>=3.9,<4",
    "cvxpy": "cvxpy>=1.6,<2",
    "osqp": "osqp>=1,<2",
}
missing = [
    spec for name, spec in packages.items() if importlib.util.find_spec(name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import numpy as np
import scipy
from scipy.linalg import solve_discrete_are, expm
from scipy.signal import cont2discrete
import matplotlib
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path
import tempfile
import urllib.request
from matplotlib import font_manager

# Colab과 로컬에서 동일하게 한글 그래프를 표시하기 위한 공개 글꼴.
font_dir = Path(tempfile.gettempdir()) / "mpc_korean_fonts"
font_dir.mkdir(exist_ok=True)
font_path = font_dir / "NanumGothic-Regular.ttf"
if not font_path.exists():
    url = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    with urllib.request.urlopen(url, timeout=60) as response:
        font_path.write_bytes(response.read())
font_manager.fontManager.addfont(str(font_path))
korean_font = font_manager.FontProperties(fname=font_path).get_name()

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update(
    {
        "figure.figsize": (9, 3.8),
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 11,
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": ["DejaVu Sans", korean_font],
        "mathtext.fontset": "dejavusans",
        "axes.unicode_minus": False,
    }
)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
CHECKS = {}


def finish_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()


def riccati(A, B, Q, R, Pf, N):
    # 부호 관례: u = K x. K에 음의 부호를 포함한다.
    Ps = [None] * (N + 1)
    Ks = [None] * N
    Ps[N] = Pf.copy()
    for k in range(N - 1, -1, -1):
        Pnext = Ps[k + 1]
        Ks[k] = -np.linalg.solve(R + B.T @ Pnext @ B, B.T @ Pnext @ A)
        Ps[k] = Q + A.T @ Pnext @ A + A.T @ Pnext @ B @ Ks[k]
        Ps[k] = (Ps[k] + Ps[k].T) / 2
    return Ps, Ks


def simulate_linear(A, B, policy, x0, steps):
    xs = [np.array(x0, dtype=float)]
    us = []
    for k in range(steps):
        u = np.atleast_1d(policy(xs[-1], k))
        us.append(u)
        xs.append(A @ xs[-1] + B @ u)
    return np.array(xs), np.array(us)


def checked_solve(problem):
    problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, max_iter=20000)
    if problem.status != cp.OPTIMAL or not np.isfinite(problem.value):
        raise RuntimeError(f"최적화 실패: {problem.status}")
    residual = max(
        (float(np.max(c.violation())) for c in problem.constraints), default=0.0
    )
    if not np.isfinite(residual) or residual > 2e-6:
        raise RuntimeError(f"제약 잔차 초과: {residual:.3g}")
    return float(problem.value)


In [ ]:
from scipy.optimize import least_squares
from scipy.special import logsumexp
from time import perf_counter

CHECKS = {}


def linear_data(T=30, seed=4):
    A = np.array([[1.0, 0.2], [0.0, 0.94]])
    C = np.array([[1.0, 0.0]])
    Q = np.diag([0.003, 0.001])
    R = np.array([[0.04]])
    m0 = np.zeros(2)
    P0 = np.diag([1.0, 0.5])
    rng = np.random.default_rng(seed)
    x = np.zeros((T + 1, 2))
    x[0] = [1.2, -0.25]
    y = np.zeros((T, 1))
    for k in range(T):
        y[k] = C @ x[k] + rng.multivariate_normal(np.zeros(1), R)
        x[k + 1] = A @ x[k] + rng.multivariate_normal(np.zeros(2), Q)
    return A, C, Q, R, m0, P0, x, y


def linear_kf(A, C, Q, R, m0, P0, y, B=None, controls=None):
    m = m0.copy()
    P = P0.copy()
    pred = [m.copy()]
    priors = [P.copy()]
    post = []
    posts = []
    for k, yk in enumerate(y):
        L = np.linalg.solve(C @ P @ C.T + R, C @ P).T
        m = m + L @ (yk - C @ m)
        I = np.eye(len(m)) - L @ C
        P = I @ P @ I.T + L @ R @ L.T
        P = (P + P.T) / 2
        post.append(m.copy())
        posts.append(P.copy())
        m = A @ m + (B @ np.atleast_1d(controls[k]) if B is not None else 0)
        P = A @ P @ A.T + Q
        pred.append(m.copy())
        priors.append(P.copy())
    return np.array(pred), np.array(priors), np.array(post), np.array(posts)


def whitener(covariance, name):
    """W @ residual의 제곱합이 residual.T @ covariance^-1 @ residual."""
    covariance = np.asarray(covariance, dtype=float)
    if (
        covariance.ndim != 2
        or covariance.shape[0] != covariance.shape[1]
        or not np.all(np.isfinite(covariance))
    ):
        raise ValueError(f"{name}: 유한한 정사각 공분산이 필요합니다.")
    if not np.allclose(covariance, covariance.T, rtol=1e-10, atol=1e-12):
        raise ValueError(f"{name}: 공분산은 대칭이어야 합니다.")
    try:
        factor = np.linalg.cholesky(covariance)
    except np.linalg.LinAlgError as error:
        raise ValueError(f"{name}: 양의 정부호 공분산이 필요합니다.") from error
    return scipy.linalg.solve_triangular(factor, np.eye(len(factor)), lower=True)


def window_fit(
    A, C, Q, R, y, arrival=None, B=None, controls=None, positive=False, prediction=True
):
    A = np.asarray(A, dtype=float)
    C = np.asarray(C, dtype=float)
    y = np.asarray(y, dtype=float)
    if (
        A.ndim != 2
        or A.shape[0] != A.shape[1]
        or C.ndim != 2
        or C.shape[1] != A.shape[0]
    ):
        raise ValueError("A와 C의 상태 차원이 맞아야 합니다.")
    if (
        y.ndim != 2
        or len(y) < 1
        or y.shape[1] != C.shape[0]
        or not all(np.all(np.isfinite(v)) for v in (A, C, y))
    ):
        raise ValueError(
            "측정은 비어 있지 않은 (시점, 측정 차원) 유한 배열이어야 합니다."
        )
    M = len(y)
    n = A.shape[0]
    J = M if prediction else M - 1
    Wq = whitener(Q, "Q")
    Wr = whitener(R, "R")
    if Wq.shape != (n, n) or Wr.shape != (C.shape[0], C.shape[0]):
        raise ValueError("잡음 공분산 차원이 맞지 않습니다.")
    if (B is None) != (controls is None):
        raise ValueError("B와 실제 입력 이력은 함께 제공해야 합니다.")
    if B is not None:
        B = np.asarray(B, dtype=float)
        controls = np.asarray(controls, dtype=float)
        if (
            B.ndim != 2
            or B.shape[0] != n
            or len(controls) != J
            or not np.all(np.isfinite(controls))
            or not np.all(np.isfinite(B))
        ):
            raise ValueError("모델의 전이 수와 실제 입력 이력이 맞아야 합니다.")
        controls = controls.reshape(J, B.shape[1])
    z = cp.Variable((n, J + 1))
    cost = 0
    cons = []
    if arrival is not None:
        m, P = arrival
        m = np.asarray(m, dtype=float)
        Wp = whitener(P, "P")
        if m.shape != (n,) or Wp.shape != (n, n) or not np.all(np.isfinite(m)):
            raise ValueError("사전 평균·공분산 차원이 맞지 않습니다.")
        cost += cp.sum_squares(Wp @ (z[:, 0] - m))
    for j in range(J):
        known = B @ controls[j] if B is not None else np.zeros(n)
        cost += cp.sum_squares(Wq @ (z[:, j + 1] - A @ z[:, j] - known))
    for j in range(M):
        cost += cp.sum_squares(Wr @ (y[j] - C @ z[:, j]))
    if positive:
        cons.append(z >= 0)
    problem = cp.Problem(cp.Minimize(cost), cons)
    checked_solve(problem)
    if not np.all(np.isfinite(z.value)):
        raise RuntimeError("추정 결과가 유한하지 않습니다.")
    return z.value.T.copy(), float(problem.value)


# Monotone nonlinear observation: one common data set for EKF / UKF / MHE.
def nf(x):
    return 0.9 * x + 0.1 * np.tanh(x)


def nh(x):
    return x + 0.15 * x**3


def ndf(x):
    return 0.9 + 0.1 / np.cosh(x) ** 2


def ndh(x):
    return 1 + 0.45 * x * x


def nonlinear_data(T=45, seed=24):
    q = 0.03**2
    r = 0.08**2
    rng = np.random.default_rng(seed)
    x = np.zeros(T + 1)
    x[0] = 1.2
    y = np.zeros(T)
    for k in range(T):
        y[k] = nh(x[k]) + rng.normal(0, np.sqrt(r))
        x[k + 1] = nf(x[k]) + rng.normal(0, np.sqrt(q))
    return x, y, q, r


def sigma_points(m, P):
    # Scaled unscented transform: n=1, alpha=.7, beta=2, kappa=0.
    if not np.isfinite(m) or not np.isfinite(P) or P <= 0:
        raise ValueError("시그마 점의 평균은 유한하고 분산은 양수여야 합니다.")
    lam = 0.7**2 - 1
    wm = np.array([lam / (1 + lam), 1 / (2 * (1 + lam)), 1 / (2 * (1 + lam))])
    wc = wm.copy()
    wc[0] += 1 - 0.7**2 + 2
    z = m + np.array([0.0, 1.0, -1.0]) * np.sqrt((1 + lam) * P)
    return z, wm, wc


def nonlinear_filter(y, q, r, kind="EKF", f=nf, h=nh, df=ndf, dh=ndh, m0=0.2, P0=0.6):
    if kind not in ("EKF", "UKF"):
        raise ValueError("필터 종류는 EKF 또는 UKF여야 합니다.")
    if (
        not np.all(np.isfinite([q, r, m0, P0]))
        or q < 0
        or min(r, P0) <= 0
        or not np.all(np.isfinite(y))
    ):
        raise ValueError("유한한 데이터와 유효한 잡음·사전 분산이 필요합니다.")
    m = m0
    P = P0
    pred = [m]
    priors = [P]
    posterior = []
    for yk in y:
        if kind == "EKF":
            H = dh(m)
            S = H * P * H + r
            L = P * H / S
            mp = m + L * (yk - h(m))
            PP = (1 - L * H) ** 2 * P + L * L * r
            P = df(mp) ** 2 * PP + q
            m = f(mp)
        else:
            sp, wm, wc = sigma_points(m, P)
            obs = h(sp)
            mean = wm @ obs
            S = wc @ ((obs - mean) ** 2) + r
            cross = wc @ ((sp - m) * (obs - mean))
            L = cross / S
            mp = m + L * (yk - mean)
            PP = P - L * L * S
            assert PP > 0 and np.isfinite(PP)
            sp, wm, wc = sigma_points(mp, PP)
            zz = f(sp)
            m = wm @ zz
            P = wc @ ((zz - m) ** 2) + q
        assert P > 0 and np.isfinite(m)
        posterior.append(mp)
        pred.append(m)
        priors.append(P)
    return np.array(pred), np.array(priors), np.array(posterior)


print("제4장 공통 함수 준비 완료. 각 실습은 다른 실습 결과에 의존하지 않습니다.")


## 4.1 Introduction — 개요

**교재 인쇄 pp.269**

상태 추정은 잡음이 있는 동역학과 측정값을 함께 사용하여 직접 측정하지 못한 상태를 추론합니다. 추정기의 모델 잔차와 측정 잔차를 구분하며, 추정값을 감시나 피드백 제어에 사용할 수 있습니다. 교재는 먼저 알려진 입력이 없는 추정 문제를 다루고 §4.6에서 제어와 연결합니다.

## 4.2 Full Information Estimation — 전체 정보 추정

**교재 인쇄 pp.269–278**

전체 정보 추정(FIE)은 처음부터 현재까지의 모든 측정과 초기 사전 정보를 사용합니다. 초기 상태 편차, 과정 외란, 측정 잔차의 비용을 최소화하고 모델로 상태들을 연결합니다. 선형·무제약·가우스 조건에서 이차 비용의 최적 상태는 조건부 평균과 일치합니다. 비선형 안정성 분석에서는 검출가능성에 해당하는 i-IOSS와 잡음·비용에 대한 가정을 함께 사용합니다.

$$
J_T=\|\xi_0-m_0^-\|_{(P_0^-)^{-1}}^2+\sum_{j=0}^{T-1}\left(\|\xi_{j+1}-A\xi_j\|_{Q_w^{-1}}^2+\|y_j-C\xi_j\|_{R_v^{-1}}^2\right)
$$

**i-IOSS와 추정 오차:** 두 경로의 차이를 $\Delta x,\Delta w,\Delta y$로 쓰면
$$\|\Delta x_k\|\le\beta(\|\Delta x_0\|,k)+\gamma_w(\|\Delta w\|_{0:k-1})+\gamma_y(\|\Delta y\|_{0:k-1}).$$
$\beta\in\mathcal{KL}$은 초기 차이의 영향이 시간에 따라 줄어듦을, $\gamma_w,\gamma_y\in\mathcal K$는 외란·출력 차이가 상태 차이에 미치는 영향을 나타냅니다. 구간 노름은 해당 구간의 최대 크기입니다. 측정과 외란의 차이가 작아도 숨은 불안정 상태가 달라질 수 있다면 이 조건이 성립하지 않습니다. 유한 구간에서 초기상태가 유일하게 복원되는 관측가능성보다 약한 개념입니다. 잡음이 계속 존재하면 잔여 오차를 허용하는 분석이 필요합니다.


In [ ]:
def demo_fie():
    A, C, Q, R, m0, P0, x, y = linear_data()
    pred, Ps, post, PP = linear_kf(A, C, Q, R, m0, P0, y)
    est = []
    differences = []
    for T in range(1, len(y) + 1):
        z, _ = window_fit(A, C, Q, R, y[:T], (m0, P0))
        est.append(z[-1])
        differences.append(np.max(np.abs(z[-1] - pred[T])))
    assert max(differences) < 2e-6
    CHECKS["FIE–칼만 예측 최대 차이"] = float(max(differences))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = np.arange(len(x))
    ax[0].plot(tt, x[:, 0], label="실제 상태")
    ax[0].plot(tt, pred[:, 0], label="칼만 예측")
    ax[0].plot(tt[1:], np.array(est)[:, 0], "o", ms=3, label="FIE")
    ax[0].scatter(tt[:-1], y[:, 0], s=8, alpha=0.4, label="측정")
    ax[0].set(xlabel="상태 시점 T", ylabel="첫 상태")
    ax[0].legend(fontsize=8)
    ax[1].plot(range(1, len(x)), differences, "o-")
    ax[1].set(xlabel="상태 시점 T", ylabel="FIE–칼만 최대 성분 차이")
    finish_figure("fie_kalman")


demo_fie()


**보충 실습 — 갑자기 달라진 측정은 언제 추정에 반영될까요?**

같은 상태 $x_k$의 예측 $m_k^-$는 $y_k$를 아직 사용하지 않습니다. 필터링 $m_k^+$는 이를 사용합니다. 한 시점의 측정만 바꾼 두 데이터에서 예측은 그 시점까지 같지만 필터링은 즉시 달라져야 합니다. 이 실험은 실제 상태 변화와 센서 이상을 구분하는 탐지기는 아니며, **사용한 정보의 시간 경계**만 검증합니다.

In [ ]:
def demo_information_time():
    A = np.array([[1.0]])
    C = A.copy()
    Q = np.array([[0.01]])
    R = np.array([[0.04]])
    y = np.zeros((16, 1))
    changed = y.copy()
    changed[7] = 1
    base = linear_kf(A, C, Q, R, np.zeros(1), np.eye(1), y)
    perturbed = linear_kf(A, C, Q, R, np.zeros(1), np.eye(1), changed)
    pred, P, post, _ = perturbed
    assert np.allclose(pred[:8], base[0][:8]) and post[7, 0] > 0 and pred[8, 0] > 0
    CHECKS["현재 측정 변경 전 예측 차이"] = float(
        np.max(np.abs(pred[:8] - base[0][:8]))
    )
    CHECKS["현재 측정에 대한 필터링 반응"] = float(post[7, 0])
    plt.figure()
    plt.step(range(16), changed[:, 0], where="mid", label="측정 y(k)", alpha=0.5)
    plt.plot(range(16), pred[:-1, 0], "o-", label="예측: y(k) 미사용")
    plt.plot(range(16), post[:, 0], "s-", label="필터링: y(k) 포함")
    plt.axvline(7, color="gray", ls=":")
    plt.xlabel("같은 상태의 시점 k")
    plt.ylabel("측정과 추정")
    plt.legend()
    finish_figure("information_time")


demo_information_time()


**보충 실습 — 작은 RMSE만으로 공분산을 믿을 수 있을까요?**

추정 평균의 정확도와 추정기가 보고하는 불확실성의 정확도는 다른 검사입니다. 스칼라 선형 가우스 모델 $x_{k+1}=0.9x_k+w_k$, $y_k=x_k+v_k$에서 실제 $Q=0.01$, $R=0.09$를 사용합니다. $x_0\sim N(0,1)$이며 초기 사전도 동일합니다. 서로 독립인 12,000개 경로의 **같은 측정 데이터**에 정확한 $R$과 과소 추정한 $R/9$를 사용하는 두 KF를 적용합니다.

현재 측정 전 혁신 $\nu_k=y_k-m_k^-$와 보고된 혁신 분산 $S_k=P_k^-+R_{\rm assumed}$에 대해 $\mathrm{NIS}=\nu_k^2/S_k$를, 현재 측정 후 오차에 대해 $\mathrm{NEES}=(x_k-m_k^+)^2/P_k^+$를 계산합니다. 정확한 모델·사전·독립 가우스 잡음에서는 두 통계량이 각각 $\chi^2_1$ 분포이고 기대값은 1입니다. $M$개 **독립 경로의 같은 마지막 시점**에서 평균을 내므로 $M\overline{\mathrm{NIS}}$, $M\overline{\mathrm{NEES}}$ 각각의 기준 분포는 $\chi^2_M$입니다. 서로 다른 시점의 상관된 오차를 독립 표본으로 세지 않습니다. 두 통계량끼리 독립이라고 가정하지도 않습니다.

잘못된 잡음 가중치에서도 실제 오차 분산 $V$는 별도로 계산할 수 있습니다:
$$V_k^+=(1-L_k)^2V_k^-+L_k^2R_{\rm true},\qquad V_{k+1}^-=0.9^2V_k^++Q.$$
따라서 정확한 비교값은 $E[\mathrm{NIS}]=(V_k^-+R_{\rm true})/S_k$, $E[\mathrm{NEES}]=V_k^+/P_k^+$입니다. 이 재귀로 시뮬레이션을 독립 검산하고, 보고된 95% 구간 $m_k^+\pm1.96\sqrt{P_k^+}$의 실제 포함률도 계산합니다. RMSE가 작아 보이더라도 구간이 너무 좁을 수 있습니다.

NIS는 실제 상태를 모르는 운영 데이터에서도 계산할 수 있지만 NEES와 실제 포함률에는 참 상태가 필요합니다. 큰 NIS가 센서 잡음 오설정만을 특정하지는 않습니다. 모델 오차·편향·이상치 등도 원인이 될 수 있으며, 아래 카이제곱 기준을 일반 비선형·제약 MHE에 그대로 적용하면 안 됩니다.


In [ ]:
def demo_covariance_calibration():
    from scipy.stats import chi2, norm

    runs, steps = 12000, 30
    a, q, r_true = 0.9, 0.01, 0.09
    rng = np.random.default_rng(4404)
    truth = rng.normal(size=runs)
    means = np.zeros((2, runs))
    reported = np.ones(2)
    actual = np.ones(2)
    r_assumed = np.array([r_true, r_true / 9])
    for k in range(steps):
        y = truth + rng.normal(0, np.sqrt(r_true), runs)
        innovation = y[None, :] - means
        S = reported + r_assumed
        gains = reported / S
        nis = np.mean(innovation**2 / S[:, None], axis=1)
        expected_nis = (actual + r_true) / S
        means += gains[:, None] * innovation
        reported = (1 - gains)**2 * reported + gains**2 * r_assumed
        actual = (1 - gains)**2 * actual + gains**2 * r_true
        if k != steps - 1:
            truth = a * truth + rng.normal(0, np.sqrt(q), runs)
            means = a * means
            reported = a**2 * reported + q
            actual = a**2 * actual + q
    errors = truth[None, :] - means
    rmse = np.sqrt(np.mean(errors**2, axis=1))
    nees = np.mean(errors**2 / reported[:, None], axis=1)
    expected_nees = actual / reported
    z = norm.ppf(0.975)
    coverage = np.mean(np.abs(errors) <= z * np.sqrt(reported[:, None]), axis=1)
    expected_coverage = 2 * norm.cdf(z * np.sqrt(reported / actual)) - 1
    # Each mean uses independent paths at one time, not correlated time samples.
    interval = chi2.ppf([0.005, 0.995], runs) / runs
    assert interval[0] < nis[0] < interval[1]
    assert interval[0] < nees[0] < interval[1]
    assert np.allclose(nis, expected_nis, rtol=0.05)
    assert np.allclose(nees, expected_nees, rtol=0.05)
    assert np.allclose(rmse, np.sqrt(actual), rtol=0.03)
    assert np.all(np.abs(coverage - expected_coverage) < 0.015)
    assert coverage[1] < 0.85 and expected_nees[1] > 2
    labels = ["정확한 R", "R/9 사용"]
    pos = np.arange(2)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    axes[0].bar(pos, rmse, label="표본 RMSE")
    axes[0].plot(pos, np.sqrt(actual), "ko", label="실제 분산 재귀")
    axes[0].set(xticks=pos, xticklabels=labels, title="상태 추정 오차")
    axes[1].bar(pos - 0.18, nis, 0.36, label="NIS")
    axes[1].bar(pos + 0.18, nees, 0.36, label="NEES")
    axes[1].plot(pos - 0.18, expected_nis, "ko")
    axes[1].plot(pos + 0.18, expected_nees, "ko", label="정확한 기대값")
    axes[1].axhspan(*interval, color="green", alpha=0.18, label="정확 모델 99% 기준")
    axes[1].set(xticks=pos, xticklabels=labels, title="보고 공분산의 일관성")
    axes[2].bar(pos, coverage, label="실제 포함률")
    axes[2].plot(pos, expected_coverage, "ko", label="가우스 해석값")
    axes[2].axhline(0.95, color="black", ls="--", label="목표 95%")
    axes[2].set(xticks=pos, xticklabels=labels, ylim=(0, 1.05), title="보고된 95% 구간")
    for ax in axes:
        ax.legend(fontsize=8)
    finish_figure("covariance_calibration")
    for j, name in enumerate(["correct", "understated"]):
        CHECKS[f"calibration_{name}_rmse"] = float(rmse[j])
        CHECKS[f"calibration_{name}_nis"] = float(nis[j])
        CHECKS[f"calibration_{name}_nees"] = float(nees[j])
        CHECKS[f"calibration_{name}_coverage"] = float(coverage[j])
    print("정확 R / R/9: RMSE, NIS, NEES, 포함률")
    print(np.column_stack([rmse, nis, nees, coverage]))
    print("독립 경로 평균 NIS 또는 NEES의 정확 모델 99% 기준:", interval)


demo_covariance_calibration()


### 4.2.1 State Estimation as Optimal Control of Estimate Error — 추정 오차의 최적 제어 관점

**교재 인쇄 pp.278–281**

추정과 조절은 최적화 구조가 비슷하지만 값 함수의 역할까지 같지는 않습니다. FIE는 시간이 지날수록 측정과 비용 항을 추가하므로 최적 비용이 증가할 수 있습니다. 이 비용 자체를 조절 MPC의 감소하는 리아푸노프 함수처럼 해석하면 안 됩니다. 추정 오차의 안정성은 별도 가정과 오차 경계로 분석합니다.

In [ ]:
def demo_cost():
    A, C, Q, R, m0, P0, x, y = linear_data(T=24)
    costs = []
    errors = []
    for T in range(1, len(y) + 1):
        z, value = window_fit(A, C, Q, R, y[:T], (m0, P0))
        costs.append(value)
        errors.append(np.linalg.norm(z[-1] - x[T]))
    assert np.min(np.diff(costs)) >= -2e-6
    CHECKS["FIE 누적 최적 비용 증가량"] = float(costs[-1] - costs[0])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = range(1, len(y) + 1)
    ax[0].plot(tt, costs, "o-")
    ax[0].set(xlabel="상태 시점 T", ylabel="누적 최적 잔차 비용")
    ax[1].plot(tt, errors, "o-")
    ax[1].set(xlabel="상태 시점 T", ylabel="추정 오차 노름")
    finish_figure("estimation_cost")


demo_cost()


### 4.2.2 Duality of Linear Estimation and Regulation — 선형 추정과 조절의 쌍대성

**교재 인쇄 pp.281–283**

선형 추정의 리카티 방정식은 전치된 모델의 LQR 문제와 쌍대 관계를 갖습니다. 예측형 관측기 이득 Le와 측정 갱신 이득 L을 구분해야 합니다. 검출가능성과 안정화가능성 등 조건 아래 안정한 정상상태 오차계를 얻습니다.

$$
P=APA^\top+Q_w-APC^\top(CPC^\top+R_v)^{-1}CPA^\top,\qquad L_e=-K_{\mathrm{dual}}^\top
$$

In [ ]:
def demo_duality():
    A, C, Q, R, *_ = linear_data()
    P = solve_discrete_are(A.T, C.T, Q, R)
    L = np.linalg.solve(C @ P @ C.T + R, C @ P).T
    Le = A @ L
    Kdual = -np.linalg.solve(R + C @ P @ C.T, C @ P @ A.T)
    assert np.max(np.abs(Le + Kdual.T)) < 1e-12
    radius = max(abs(np.linalg.eigvals(A - Le @ C)))
    assert radius < 1
    Piter = np.eye(2)
    for _ in range(500):
        Piter = (
            A @ Piter @ A.T
            + Q
            - A @ Piter @ C.T @ np.linalg.solve(C @ Piter @ C.T + R, C @ Piter @ A.T)
        )
    assert np.max(np.abs(Piter - P)) < 1e-9
    CHECKS["추정–LQR 쌍대 이득 최대 차이"] = float(np.max(np.abs(Le + Kdual.T)))
    CHECKS["정상상태 추정 오차계 스펙트럴 반지름"] = float(radius)
    print("예측형 관측기 고유값:", np.linalg.eigvals(A - Le @ C))


demo_duality()


**보충 실습 — 관측되지 않는 모드가 안정하면 검출가능할 수 있습니다**

$A=\mathrm{diag}(0.9,\lambda)$, $C=(1,0)$의 두 초기상태 $(1,1)$과 $(1,-1)$는 완전히 같은 출력 경로를 만듭니다. $\lambda=0.7$이면 숨은 상태 차이가 저절로 줄어 검출가능하지만, $\lambda=1.1$이면 그 차이가 커져 검출가능하지 않습니다. 어떤 추정 알고리즘을 써도 이 출력만으로 두 초기 상태를 유일하게 구별할 수 없습니다.

이는 §4.2의 i-IOSS 설명과 §4.3.1의 유한 구간 관측가능성을 연결하는 보충 예입니다. **초기 상태의 유일한 복원**과 **현재 상태 오차의 점근 감소**는 다른 요구입니다.

In [ ]:
def demo_detectability():
    T = 24
    t = np.arange(T + 1)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for lam in [0.7, 1.1]:
        A = np.diag([0.9, lam])
        C = np.array([[1.0, 0.0]])
        paths = np.array(
            [
                [np.linalg.matrix_power(A, k) @ np.array([1.0, s]) for k in t]
                for s in [1, -1]
            ]
        )
        assert np.allclose(paths[0, :, 0], paths[1, :, 0])
        O = np.vstack([C @ np.linalg.matrix_power(A, k) for k in range(4)])
        assert np.linalg.matrix_rank(O) == 1
        gap = np.linalg.norm(paths[0] - paths[1], axis=1)
        assert np.allclose(gap, 2 * lam**t)
        ax[0].plot(t, paths[0, :, 0], label=f"숨은 고유값 {lam}")
        ax[1].semilogy(t, gap, label=f"숨은 고유값 {lam}")
        CHECKS[f"숨은 고유값 {lam}의 마지막 상태 차이"] = float(gap[-1])
    ax[0].set(xlabel="시간 단계", ylabel="두 초기상태에서 같은 출력")
    ax[1].set(xlabel="시간 단계", ylabel="구별할 수 없는 상태 차이")
    for a in ax:
        a.legend()
    finish_figure("detectability")


demo_detectability()


## 4.3 Moving Horizon Estimation — 이동 구간 추정

**교재 인쇄 pp.283–300**

MHE는 최근 N개의 측정을 사용하고 그 이전 정보를 구간 시작점의 사전 비용에 요약합니다. 초기 데이터가 N개보다 적을 때는 전체 정보 문제로 시작합니다. 구간 길이가 고정되어도 정확한 비선형 도착 비용의 표현이 간단해지는 것은 아닙니다.

$$
J_T=\Gamma_s(\xi_s)+\sum_{j=s}^{T-1}\left(\|\xi_{j+1}-A\xi_j\|_{Q_w^{-1}}^2+\|y_j-C\xi_j\|_{R_v^{-1}}^2\right),\quad s=\max(0,T-N)
$$

### 4.3.1 Zero Prior Weighting — 사전 가중치가 없는 추정

**교재 인쇄 pp.283–287**

사전 비용이 0이면 창 밖의 정보를 완전히 버립니다. 그러므로 최근 측정만으로 상태를 재구성할 수 있는 관측가능성이 중요합니다. 검출가능성만으로는 관측되지 않는 초기 상태의 유일한 복원이 보장되지 않습니다. 아래 실습은 과정 외란이 없는 선형 관측가능성 계산으로 이 차이를 보여 줍니다.

In [ ]:
def demo_zero_prior():
    A, C, *_ = linear_data()
    initial = np.array([1.2, -0.25])
    small = []
    errors = []
    rng = np.random.default_rng(7)
    noise = rng.normal(0, 0.02, 15)
    for N in range(1, 16):
        O = np.vstack([C @ np.linalg.matrix_power(A, j) for j in range(N)])
        rank = np.linalg.matrix_rank(O)
        assert rank == (1 if N == 1 else 2)
        exact = np.linalg.lstsq(O, O @ initial, rcond=None)[0]
        if N >= 2:
            assert np.linalg.norm(exact - initial) < 1e-10
        estimated = np.linalg.lstsq(O, O @ initial + noise[:N], rcond=None)[0]
        small.append(np.linalg.eigvalsh(O.T @ O)[0])
        errors.append(np.linalg.norm(estimated - initial))
    Ad = np.diag([1.1, 0.8])
    Cd = np.array([[1.0, 0.0]])
    assert (
        np.linalg.matrix_rank(
            np.vstack([Cd @ np.linalg.matrix_power(Ad, j) for j in range(10)])
        )
        == 1
    )
    CHECKS["관측가능성 확보 최소 측정 수"] = 2
    CHECKS["관측 불가능한 안정 모드"] = 0.8
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(range(1, 16), small, "o-")
    ax[0].set(xlabel="측정 수 N", ylabel="관측 그램 행렬 최소 고유값")
    ax[1].plot(range(1, 16), errors, "o-")
    ax[1].set(xlabel="측정 수 N", ylabel="초기 상태 복원 오차")
    finish_figure("zero_prior")


demo_zero_prior()


### 4.3.2 Nonzero Prior Weighting — 사전 가중치를 사용하는 추정

**교재 인쇄 pp.287–294**

정확한 도착 비용은 창 밖의 모든 정보를 창 시작 상태의 함수로 요약하므로 MHE와 FIE가 같습니다. 선형 무제약 가우스 문제에서는 창 시작점의 칼만 예측 평균과 공분산으로 이 비용을 정확히 표현할 수 있습니다. 이 사전 정보는 반드시 창에 포함되지 않은 측정에서만 계산해야 합니다. 비선형이나 제약 문제에서 같은 이차식을 사용하면 일반적으로 근사입니다.

$$
\Gamma_s(\xi_s)=\|\xi_s-m_s^-\|_{(P_s^-)^{-1}}^2+\mathrm{const.}
$$

In [ ]:
def demo_arrival():
    A, C, Q, R, m0, P0, x, y = linear_data(T=24)
    pred, Ps, _, _ = linear_kf(A, C, Q, R, m0, P0, y)
    diffs = {}
    zero = []
    for N in [1, 3, 8]:
        dd = []
        for T in range(1, len(y) + 1):
            s = max(0, T - N)
            z, _ = window_fit(A, C, Q, R, y[s:T], (pred[s], Ps[s]))
            dd.append(np.linalg.norm(z[-1] - pred[T]))
        diffs[N] = dd
        assert max(dd) < 3e-6
    for T in range(1, len(y) + 1):
        s = max(0, T - 3)
        z, _ = window_fit(A, C, Q, R, y[s:T], (m0, P0) if T <= 3 else None)
        zero.append(z[-1])
    CHECKS["정확한 도착 비용 MHE–KF 최대 차이"] = float(
        max(max(v) for v in diffs.values())
    )
    CHECKS["사전 없는 N3과 KF의 최대 차이"] = float(
        np.max(np.abs(np.array(zero) - pred[1:]))
    )
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = range(1, len(x))
    for N, dd in diffs.items():
        ax[0].plot(tt, dd, label=f"창 길이 {N}")
    ax[0].set(xlabel="상태 시점 T", ylabel="MHE–칼만 예측 차이")
    ax[0].legend()
    ax[1].plot(x[:, 1], label="실제 두 번째 상태")
    ax[1].plot(pred[:, 1], label="정확한 도착 비용")
    ax[1].plot(tt, np.array(zero)[:, 1], label="사전 없음 N=3")
    ax[1].set(xlabel="상태 시점 T", ylabel="두 번째 상태")
    ax[1].legend(fontsize=8)
    finish_figure("arrival_cost")


demo_arrival()


**보충 실습 — 과거를 최소화해 없애면 도착 비용이 됩니다**

스칼라 모델 $x^+=1.1x+w$, $y=x+v$에서 시점 $s=4$의 끝 상태를 $q$로 고정하고, 과거 상태 $x_0,\ldots,x_{s-1}$를 최소화합니다. 과거 전체 이차 비용의 Hessian을 분할하면
$$H_{\mathrm{arr}}=H_{qq}-H_{qp}H_{pp}^{-1}H_{pq}.$$
이는 Schur 보완이며, 코드에서는 역행렬을 만들지 않고 선형계를 풉니다. 얻은 비용곡선은 $\Gamma_s(q)-\min\Gamma_s=(q-m_s^-)^2/P_s^-$와 일치합니다. 도착 비용은 임의의 정규화 항이 아니라 **창 밖 정보를 요약한 값 함수**입니다. 여기서는 선형·무제약 조건의 정확한 이차식이며 제약·비선형에서는 일반적으로 근사입니다.

In [ ]:
def demo_arrival_elimination():
    a = 1.1
    q = 0.09
    r = 0.16
    p0 = 0.7
    m0 = -0.2
    y = np.array([0.4, 0.2, 0.6, 0.5])
    s = len(y)
    rows = []
    rhs = []
    row = np.zeros(s + 1)
    row[0] = 1 / np.sqrt(p0)
    rows.append(row)
    rhs.append(m0 / np.sqrt(p0))
    for j in range(s):
        row = np.zeros(s + 1)
        row[j] = -a / np.sqrt(q)
        row[j + 1] = 1 / np.sqrt(q)
        rows.append(row)
        rhs.append(0.0)
        row = np.zeros(s + 1)
        row[j] = 1 / np.sqrt(r)
        rows.append(row)
        rhs.append(y[j] / np.sqrt(r))
    D = np.array(rows)
    b = np.array(rhs)
    H = D.T @ D
    g = D.T @ b
    schur = H[-1, -1] - H[-1, :-1] @ np.linalg.solve(H[:-1, :-1], H[:-1, -1])
    linear = g[-1] - H[-1, :-1] @ np.linalg.solve(H[:-1, :-1], g[:-1])
    mean = linear / schur
    pred, Ps, *_ = linear_kf(
        np.array([[a]]),
        np.ones((1, 1)),
        np.array([[q]]),
        np.array([[r]]),
        np.array([m0]),
        np.array([[p0]]),
        y[:, None],
    )
    assert abs(mean - pred[-1, 0]) < 1e-12 and abs(1 / schur - Ps[-1, 0, 0]) < 1e-12
    optimum = np.linalg.lstsq(D, b, rcond=None)[0]
    minimum = np.sum((D @ optimum - b) ** 2)
    grid = np.linspace(mean - 1.5, mean + 1.5, 81)
    profile = []
    for endpoint in grid:
        past = np.linalg.lstsq(D[:, :-1], b - D[:, -1] * endpoint, rcond=None)[0]
        profile.append(np.sum((D @ np.r_[past, endpoint] - b) ** 2) - minimum)
    analytic = (grid - mean) ** 2 / Ps[-1, 0, 0]
    gap = np.max(np.abs(profile - analytic))
    assert gap < 1e-10
    CHECKS["과거 제거 도착 비용과 칼만 곡선 차이"] = float(gap)
    CHECKS["Schur 보완과 칼만 사전 분산 차이"] = float(abs(1 / schur - Ps[-1, 0, 0]))
    plt.figure()
    plt.plot(grid, analytic, label="칼만 사전으로 만든 도착 비용")
    plt.plot(
        grid[::4], np.array(profile)[::4], "o", label="끝 상태 고정 후 과거 최소화"
    )
    plt.xlabel("창 시작 상태 q")
    plt.ylabel("최소값을 뺀 도착 비용")
    plt.legend()
    finish_figure("arrival_elimination")


demo_arrival_elimination()


### 4.3.3 Constrained Estimation — 제약이 있는 추정

**교재 인쇄 pp.294–295**

추정의 제약은 농도 비음수처럼 실제 시스템에 성립하는 정보를 추가합니다. 제어기와 달리 추정기는 실제 시스템에 제약을 강제할 수 없습니다. 잘못된 제약은 추정 편향이나 실행 불가능성을 만들 수 있습니다. 아래의 제약 MHE는 선형 칼만 사전을 이차 근사로 사용하며 제약 FIE와의 정확한 일치를 주장하지 않습니다.

In [ ]:
def demo_constraints():
    T = 40
    A = np.array([[1.0]])
    C = np.array([[1.0]])
    Q = np.array([[0.01**2]])
    R = np.array([[0.07**2]])
    truth = 0.04 + 0.025 * np.sin(0.4 * np.arange(T + 1))
    rng = np.random.default_rng(15)
    y = (truth[:-1] + rng.normal(0, 0.07, T))[:, None]
    pred, Ps, _, _ = linear_kf(A, C, Q, R, np.array([0.04]), np.array([[0.04]]), y)
    uncon = []
    con = []
    for k in range(1, T + 1):
        s = max(0, k - 6)
        arrival = (pred[s], Ps[s])
        z, _ = window_fit(A, C, Q, R, y[s:k], arrival)
        uncon.append(z[-1, 0])
        z, _ = window_fit(A, C, Q, R, y[s:k], arrival, positive=True)
        con.append(z[-1, 0])
    assert min(truth) > 0 and min(con) >= -2e-6
    CHECKS["비음수 제약 추정의 최소값"] = float(min(con))
    CHECKS["무제약 추정의 최소값"] = float(min(uncon))
    CHECKS["제약 추정 RMSE"] = float(np.sqrt(np.mean((np.array(con) - truth[1:]) ** 2)))
    plt.figure()
    plt.plot(truth, label="실제 양수 상태")
    plt.scatter(range(T), y[:, 0], s=9, alpha=0.4, label="측정")
    plt.plot(range(1, T + 1), uncon, label="무제약 MHE")
    plt.plot(range(1, T + 1), con, label="비음수 MHE")
    plt.axhline(0, color="gray", ls=":")
    plt.xlabel("상태 시점 T")
    plt.ylabel("상태")
    plt.legend(fontsize=9)
    finish_figure("constrained_estimation")


demo_constraints()


**보충 실습 — 상관된 상태의 제약 추정은 성분별 clipping과 다릅니다**

무제약 추정 평균 $m=(-1,1)$, 공분산 $P=\begin{bmatrix}1&0.8\\0.8&1\end{bmatrix}$에 대해 $x_1\ge0$을 적용합니다. 공분산이 상관을 표현하므로 최적점은 $(0,1.8)$입니다. 첫 성분만 0으로 바꾸는 $(0,1)$은 허용되지만 목적값이 더 큽니다.

목적은 $(x-m)^TP^{-1}(x-m)$의 최소화입니다. 이 점은 제약된 가우스 사후의 **최빈값(MAP)**이며 절단 분포의 평균과는 다릅니다. 실제 상태가 이 제약을 지키는지와 별개로, 여기서는 주어진 이차 문제의 기하를 검증합니다.

In [ ]:
def demo_constraint_geometry():
    mean = np.array([-1.0, 1.0])
    P = np.array([[1.0, 0.8], [0.8, 1.0]])
    W = whitener(P, "P")
    x = cp.Variable(2)
    problem = cp.Problem(cp.Minimize(cp.sum_squares(W @ (x - mean))), [x[0] >= 0])
    checked_solve(problem)
    optimum = x.value
    clipped = np.maximum(mean, 0)
    expected = np.array([0.0, 1.8])
    assert np.allclose(optimum, expected, atol=2e-6)
    cost = lambda z: float(np.sum((W @ (z - mean)) ** 2))
    gap = cost(clipped) - cost(optimum)
    assert gap > 1
    CHECKS["성분별 clipping의 제약 비용 손실"] = gap
    gx, gy = np.meshgrid(np.linspace(-2, 1, 160), np.linspace(-1, 3, 160))
    delta = np.stack([gx - mean[0], gy - mean[1]], axis=-1)
    contour = np.sum((delta @ W.T) ** 2, axis=-1)
    plt.figure()
    plt.contour(gx, gy, contour, levels=[1, 2, 4, 8], colors="gray", linewidths=0.7)
    plt.axvspan(0, 1, color="teal", alpha=0.1, label="허용 영역 x₁ ≥ 0")
    for point, label, marker in [
        (mean, "무제약 평균", "o"),
        (clipped, "성분별 clipping", "x"),
        (optimum, "제약 최적점", "*"),
    ]:
        plt.scatter(*point, s=90, label=label, marker=marker)
    plt.xlim(-2, 1)
    plt.ylim(-1, 3)
    plt.xlabel("상태 1")
    plt.ylabel("상태 2")
    plt.legend()
    finish_figure("constraint_geometry")


demo_constraint_geometry()


### 4.3.4 Smoothing and Filtering Update — 평활화와 필터링 갱신

**교재 인쇄 pp.295–300**

필터링은 해당 시점까지의 측정으로 현재 상태를 추정하고, 평활화는 더 나중의 측정을 사용하여 과거 상태를 수정합니다. 교재의 평활화 사전 갱신에는 비용 보정이 포함됩니다. 평활화된 평균·공분산을 단순히 새 사전으로 넣고 같은 관측을 또 사용하면 정보를 중복 계산할 수 있습니다. 아래 실습은 RTS 평활화와 전체 정보 최적화의 일치를 확인하는 별도 선형 예입니다.

$$
m_{k|T-1}=m_k^++J_k(m_{k+1|T-1}-m_{k+1}^-),\quad J_k=P_k^+A^\top(P_{k+1}^-)^{-1}
$$

In [ ]:
def demo_smoothing():
    A, C, Q, R, m0, P0, x, y = linear_data()
    pred, Ps, post, PP = linear_kf(A, C, Q, R, m0, P0, y)
    smooth = pred.copy()
    SP = Ps.copy()
    for k in range(len(y) - 1, -1, -1):
        J = np.linalg.solve(Ps[k + 1], A @ PP[k]).T
        smooth[k] = post[k] + J @ (smooth[k + 1] - pred[k + 1])
        SP[k] = PP[k] + J @ (SP[k + 1] - Ps[k + 1]) @ J.T
        assert np.linalg.eigvalsh((PP[k] - SP[k] + (PP[k] - SP[k]).T) / 2).min() > -1e-9
    z, _ = window_fit(A, C, Q, R, y, (m0, P0))
    difference = np.max(np.abs(z - smooth))
    assert difference < 3e-6
    CHECKS["RTS 평활화–FIE 경로 최대 차이"] = float(difference)
    correct = 1 / (1 + 1 / 0.2)
    double_counted = 1 / (1 + 2 / 0.2)
    assert double_counted < correct
    CHECKS["같은 관측 중복 시 잘못된 분산 비율"] = double_counted / correct
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = range(len(y))
    ax[0].plot(x[:, 1], label="실제 두 번째 상태")
    ax[0].plot(tt, post[:, 1], label="필터링")
    ax[0].plot(smooth[:, 1], label="평활화")
    ax[0].set(xlabel="상태 시점 k", ylabel="두 번째 상태")
    ax[0].legend()
    ax[1].plot(tt, PP[:, 1, 1], label="필터링 분산")
    ax[1].plot(tt, SP[:-1, 1, 1], label="평활화 분산")
    ax[1].set(xlabel="과거 상태 시점 k", ylabel="두 번째 상태 불확실성")
    ax[1].legend()
    finish_figure("smoothing")


demo_smoothing()


## 4.4 Bounded Disturbances — 유계 외란

**교재 인쇄 pp.300–308**

잡음이 유계여도 0으로 수렴하지 않으면 무한 누적 잔차 비용은 계속 커질 수 있습니다. 교재는 사전 가중치와 구간 길이를 검출가능성·수축 조건에 연결하는 안정성 결과를 설명합니다. 아래 실습은 그 정리를 재현하는 MHE가 아니라, 정확한 스칼라 구간 집합 추정으로 모든 허용 잡음에 대한 포함 관계를 확인하는 보충 예입니다.

In [ ]:
def demo_bounded():
    T = 40
    a = 0.9
    q = 0.03
    r = 0.08
    rng = np.random.default_rng(18)
    x = 0.2
    lo = -0.5
    hi = 0.5
    states = [x]
    lower = [lo]
    upper = [hi]
    for k in range(T):
        y = x + rng.uniform(-r, r)
        L = max(lo, y - r)
        H = min(hi, y + r)
        assert L <= x <= H and L <= H
        x = a * x + rng.uniform(-q, q)
        lo = a * L - q
        hi = a * H + q
        assert lo - 1e-12 <= x <= hi + 1e-12
        states.append(x)
        lower.append(lo)
        upper.append(hi)
    center = (np.array(lower) + upper) / 2
    radius = (np.array(upper) - lower) / 2
    assert np.all(np.abs(np.array(states) - center) <= radius + 1e-12)
    # A hypothetical observation entirely outside the next admissible measurement range.
    bad_y = hi + r + 0.1
    assert max(lo, bad_y - r) > min(hi, bad_y + r)
    CHECKS["유계 추정 구간 최대 포함 위반량"] = float(
        max(0, np.max(np.abs(np.array(states) - center) - radius))
    )
    CHECKS["유계 추정 마지막 구간 폭"] = float(hi - lo)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = range(T + 1)
    ax[0].fill_between(tt, lower, upper, alpha=0.25, label="보장 예측 구간")
    ax[0].plot(states, label="실제 상태")
    ax[0].plot(center, "--", label="구간 중앙")
    ax[0].set(xlabel="상태 시점 T", ylabel="상태")
    ax[0].legend()
    ax[1].plot(2 * radius)
    ax[1].set(xlabel="상태 시점 T", ylabel="예측 구간 폭")
    finish_figure("bounded_estimation")


demo_bounded()


**원본 식의 수치 적용 — Example 4.40, pp.306–308**

원본은 특정 수치 모델 대신 **검출가능한 선형계 전체**에 대해 도착 비용 가중치와 구간 길이의 충분조건을 유도합니다. 이를 보충 스칼라 계수 $A=1.1,G=C=1,L=0.6$에 적용합니다. $A-LC=0.5$, $c=1,\lambda=0.6$이므로 원본의 $\|(A-LC)^j\|\le c\lambda^j$가 성립합니다.

원본의 비용에는 $1/2$가 있어 $P=Q=R=1$일 때 $a_\Gamma=a_\ell=1/2$이고, $a_d=3c(|G|+|L|)/(1-\lambda)$입니다. 식 (4.55)–(4.56)은
$$\eta\le\frac18\frac{a_\ell}{a_\Gamma a_d^2}\delta^2,\qquad N\ge\max\{1,\log(\delta/(3c))/\log\lambda\}.$$
여기서 $\delta$는 원본의 수축 설계 상수를 이 실습에서 부르는 기호입니다. 더 작은 수축 목표는 더 긴 충분 구간과 더 작은 사전 가중치를 요구합니다. 이 보수적 **충분조건을 어기면 반드시 발산한다는 뜻은 아닙니다**. 원본 정리의 분석 상수를 계산하는 실습이며 최적 구간 길이나 완성된 MHE 성능 비교는 아닙니다.

In [ ]:
def demo_book_horizon_conditions():
    c = 1.0
    lam = 0.6
    G = 1.0
    L = 0.6
    ag = al = 0.5
    ad = 3 * c * (abs(G) + abs(L)) / (1 - lam)
    delta = np.linspace(0.1, 0.8, 71)
    eta = (al / (8 * ag * ad**2)) * delta**2
    horizon = np.maximum(1, np.ceil(np.log(delta / (3 * c)) / np.log(lam))).astype(int)
    assert np.all(3 * c * lam**horizon <= delta + 1e-12)
    assert np.all(
        2 * np.sqrt(2) * ad * np.sqrt(ag / al) * np.sqrt(eta) <= delta + 1e-12
    )
    i = int(np.argmin(abs(delta - 0.2)))
    assert horizon[i] == 6
    CHECKS["원본 충분조건 delta 0.2의 구간 길이"] = int(horizon[i])
    CHECKS["원본 충분조건 delta 0.2의 사전 가중치 상한"] = float(eta[i])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].step(delta, horizon, where="mid")
    ax[0].set(xlabel="수축 설계 상수 delta", ylabel="충분 구간 길이 N")
    ax[1].plot(delta, eta)
    ax[1].set(xlabel="수축 설계 상수 delta", ylabel="충분 사전 가중치 상한 eta")
    finish_figure("book_horizon_conditions")


demo_book_horizon_conditions()


## 4.5 Other Nonlinear State Estimators — 다른 비선형 상태 추정기

**교재 인쇄 pp.308–318**

비선형 추정기는 확률 분포나 최적화 문제를 서로 다른 방식으로 근사합니다. EKF는 국소 선형화, UKF는 시그마 점의 모멘트 전파, 입자 필터는 가중 표본, MHE는 최근 경로 최적화를 사용합니다. 계산량과 정확도를 비교할 때 모델·데이터·사전 정보·정보 시점을 같게 맞춰야 합니다.

### 4.5.1 Particle Filtering — 입자 필터

**교재 인쇄 pp.308–309**

입자 필터는 가중 표본으로 사후 분포를 근사하고 유효 표본 수가 줄면 재표본화할 수 있습니다. 다봉 분포를 표현할 수 있지만 고차원에서는 많은 표본이 필요합니다. 교재는 이 판에서 상세 전개를 생략하므로 아래는 개념을 확장하는 보충 실습입니다. 제곱 측정에서는 부호가 구분되지 않아 사후 평균만 보면 분포의 두 봉우리를 놓칠 수 있습니다.

In [ ]:
def demo_particles():
    rng = np.random.default_rng(51)
    M = 8000
    particles = np.r_[rng.normal(-0.8, 0.12, M // 2), rng.normal(0.8, 0.12, M // 2)]
    weights = np.full(M, 1 / M)
    x = 0.9
    ess = []
    resampled = 0
    for k in range(18):
        y = x * x + rng.normal(0, 0.1)
        logw = (
            np.log(np.maximum(weights, 1e-300)) - 0.5 * ((y - particles**2) / 0.1) ** 2
        )
        weights = np.exp(logw - logsumexp(logw))
        assert abs(weights.sum() - 1) < 1e-12
        neff = 1 / np.sum(weights**2)
        ess.append(neff)
        assert 1 <= neff <= M + 1e-6
        if k == 17:
            break
        if neff < M / 2:
            positions = (rng.random() + np.arange(M)) / M
            cdf = np.cumsum(weights)
            cdf[-1] = 1
            indices = np.searchsorted(cdf, positions)
            particles = particles[indices]
            weights.fill(1 / M)
            resampled += 1
        particles = 0.95 * particles + rng.normal(0, 0.05, M)
        x = 0.95 * x + rng.normal(0, 0.05)
    mass = weights[particles > 0].sum()
    assert 0.2 < mass < 0.8 and resampled > 0
    CHECKS["입자 필터 양의 상태 사후 질량"] = float(mass)
    CHECKS["입자 필터 재표본화 횟수"] = resampled
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].hist(particles, weights=weights, bins=70, density=True, alpha=0.7)
    ax[0].axvline(weights @ particles, color="tab:orange", label="가중 평균")
    ax[0].set(xlabel="마지막 상태", ylabel="사후 확률 밀도")
    ax[0].legend()
    ax[1].plot(ess, "o-")
    ax[1].axhline(M / 2, color="gray", ls=":", label="재표본화 기준")
    ax[1].set(xlabel="측정 단계", ylabel="유효 표본 수")
    ax[1].legend()
    finish_figure("particle_filter")


demo_particles()


### 4.5.2 Extended Kalman Filtering — 확장 칼만 필터

**교재 인쇄 pp.309–310**

EKF는 현재 추정점에서 모델과 측정 함수를 선형화하여 공분산과 이득을 계산합니다. 평균은 비선형 모델을 통과시키지만 일반적으로 비선형 변환의 정확한 기대값은 아닙니다. 초기 오차나 비선형성이 크면 국소 근사가 부정확할 수 있으며 제약도 자동으로 반영하지 않습니다.

$$
L_k=P_k^-H_k^\top(H_kP_k^-H_k^\top+R_v)^{-1},\qquad m_k^+=m_k^-+L_k(y_k-h(m_k^-))
$$

In [ ]:
def demo_ekf_jacobians():
    grid = np.linspace(-2, 2, 41)
    step = 1e-6
    df_num = (nf(grid + step) - nf(grid - step)) / (2 * step)
    dh_num = (nh(grid + step) - nh(grid - step)) / (2 * step)
    residual = max(
        np.max(np.abs(df_num - ndf(grid))), np.max(np.abs(dh_num - ndh(grid)))
    )
    assert residual < 1e-8
    x, y, q, r = nonlinear_data()
    m, P, _ = nonlinear_filter(y, q, r)
    assert np.all(P > 0)
    CHECKS["비선형 야코비안 유한차분 최대 차이"] = float(residual)
    print("EKF 예측 공분산 최소값:", P.min())


demo_ekf_jacobians()


### 4.5.3 Unscented Kalman Filtering — 무향 칼만 필터

**교재 인쇄 pp.310–312**

UKF는 평균과 공분산을 재현하는 시그마 점을 비선형 함수에 통과시킨 뒤 가중 모멘트를 계산합니다. 야코비안은 필요 없지만 점 배치와 가중치, 공분산 수치 안정성을 확인해야 합니다. 시그마 점 근사는 모든 비선형 분포를 정확히 표현하거나 항상 EKF보다 좋다는 뜻은 아닙니다.

In [ ]:
def demo_sigma():
    m = 1.0
    P = 0.6
    sp, wm, wc = sigma_points(m, P)
    assert (
        abs(wm.sum() - 1) < 1e-12
        and abs(wm @ sp - m) < 1e-12
        and abs(wc @ ((sp - m) ** 2) - P) < 1e-12
    )
    exact = m + 0.15 * (m**3 + 3 * m * P)
    ut = wm @ nh(sp)
    ekf = nh(m)
    assert abs(ut - exact) < 1e-12
    rng = np.random.default_rng(16)
    yy = rng.normal(size=20)
    args = dict(
        f=lambda x: 0.9 * x,
        h=lambda x: x,
        df=lambda x: 0.9 + 0 * x,
        dh=lambda x: 1 + 0 * x,
    )
    me, pe, _ = nonlinear_filter(yy, 0.01, 0.1, kind="EKF", **args)
    mu, pu, _ = nonlinear_filter(yy, 0.01, 0.1, kind="UKF", **args)
    assert np.max(np.abs(me - mu)) < 1e-10 and np.max(np.abs(pe - pu)) < 1e-10
    CHECKS["시그마 점 비선형 평균 차이"] = abs(float(ut - exact))
    CHECKS["선형 모델 EKF–UKF 최대 차이"] = float(np.max(np.abs(me - mu)))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    xx = np.linspace(-0.5, 2.5, 200)
    ax[0].plot(xx, nh(xx), label="비선형 측정 함수")
    ax[0].scatter(sp, nh(sp), s=45, label="시그마 점")
    ax[0].set(xlabel="상태", ylabel="측정 함수 값")
    ax[0].legend()
    ax[1].bar(["평균의 함수값", "시그마 점 평균", "정확한 기대값"], [ekf, ut, exact])
    ax[1].set(ylabel="측정 예측 평균")
    finish_figure("sigma_points")


demo_sigma()


**보충 실습 — 평균이 정확해도 분산은 틀릴 수 있습니다**

바로 앞의 3차 측정 함수 $h(x)=x+0.15x^3$에서 시그마 점 평균은 가우스의 정확한 평균과 일치합니다. 그러나 분산 계산에는 $x^6$까지의 모멘트가 필요합니다. 같은 시그마 점이 이 모멘트까지 정확히 적분하지는 않습니다.

가우스 모멘트의 재귀 $M_j=mM_{j-1}+(j-1)PM_{j-2}$로 정확한 평균·분산을 계산합니다. EKF와 UKF의 분산을 비교하며, **평균 일치 검사 하나로 공분산의 정확성까지 결론내리지 않도록** 합니다.

In [ ]:
def demo_sigma_variance():
    m = 1.0
    P = 0.6
    moments = [1.0, m]
    for j in range(2, 7):
        moments.append(m * moments[-1] + (j - 1) * P * moments[-2])
    exact_mean = m + 0.15 * moments[3]
    exact_var = moments[2] + 0.3 * moments[4] + 0.0225 * moments[6] - exact_mean**2
    points, wm, wc = sigma_points(m, P)
    mean = wm @ nh(points)
    ukf_var = wc @ ((nh(points) - mean) ** 2)
    ekf_var = ndh(m) ** 2 * P
    assert abs(mean - exact_mean) < 1e-12 and abs(ukf_var - exact_var) > 0.01
    CHECKS["시그마 점 분산의 정확한 가우스 분산 대비 오차"] = float(ukf_var - exact_var)
    plt.figure()
    plt.bar(
        ["EKF 선형화", "UKF 시그마 점", "정확한 가우스 모멘트"],
        [ekf_var, ukf_var, exact_var],
    )
    plt.ylabel("측정 함수의 분산 (센서 잡음 제외)")
    finish_figure("sigma_variance")


demo_sigma_variance()


### 4.5.4 EKF, UKF, and MHE Comparison — EKF·UKF·MHE 비교

**교재 인쇄 pp.312–318**

같은 비선형 모델, 측정, 초기 사전, 잡음 가중치를 사용해 예측형 추정을 비교합니다. MHE는 창 이전 EKF 예측으로 만든 이차 사전을 사용하므로 도착 비용은 근사입니다. 비선형 최소제곱의 국소해를 사용하며 조건부 평균과의 일치나 일반적인 우월성을 주장하지 않습니다. 시간 비교는 이 실행 환경의 측정값입니다.

In [ ]:
def demo_compare():
    x, y, q, r = nonlinear_data()
    N = 6
    t = perf_counter()
    me, Pe, _ = nonlinear_filter(y, q, r, kind="EKF")
    te = perf_counter() - t
    t = perf_counter()
    mu, Pu, _ = nonlinear_filter(y, q, r, kind="UKF")
    tu = perf_counter() - t
    estimates = [0.2]
    residuals = []
    t = perf_counter()
    for T in range(1, len(y) + 1):
        s = max(0, T - N)
        m = me[s]
        P = Pe[s]
        data = y[s:T]

        def residual(z):
            return np.r_[
                (z[0] - m) / np.sqrt(P),
                (z[1:] - nf(z[:-1])) / np.sqrt(q),
                (data - nh(z[:-1])) / np.sqrt(r),
            ]

        seed = me[s : T + 1].copy()
        sol = least_squares(
            residual,
            seed,
            jac="3-point",
            ftol=1e-10,
            xtol=1e-10,
            gtol=1e-10,
            max_nfev=300,
        )
        assert sol.success, sol.message
        assert np.sum(residual(sol.x) ** 2) <= np.sum(residual(seed) ** 2) + 1e-7
        assert np.all(np.isfinite(sol.x))
        estimates.append(sol.x[-1])
        residuals.append(sol.optimality)
    tm = perf_counter() - t
    mh = np.array(estimates)
    rmse = [np.sqrt(np.mean((m[1:] - x[1:]) ** 2)) for m in [me, mu, mh]]
    for name, value in zip(["EKF", "UKF", "MHE"], rmse):
        CHECKS[name + " 비선형 예측 RMSE"] = float(value)
    CHECKS["비선형 MHE 최대 최적성 잔차"] = float(max(residuals))
    times = np.array([te, tu, tm]) / len(y) * 1000
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
    ax[0].plot(x, "k", label="실제 상태")
    for name, m in zip(["EKF", "UKF", "MHE"], [me, mu, mh]):
        ax[0].plot(m, label=name)
    ax[0].set(xlabel="상태 시점 T", ylabel="예측 상태")
    ax[0].legend(fontsize=8)
    ax[1].bar(["EKF", "UKF", "MHE"], rmse)
    ax[1].set(ylabel="예측 RMSE")
    ax[2].bar(["EKF", "UKF", "MHE"], times)
    ax[2].set(yscale="log", ylabel="단계당 계산 시간 [ms]")
    finish_figure("nonlinear_comparison")
    print("단계당 계산 시간 [ms]:", dict(zip(["EKF", "UKF", "MHE"], times)))


demo_compare()


**원본 모델 재현 — Example 4.41, pp.313–314: 압력만으로 세 농도를 추정하는 이유**

원본의 가역 반응 $A\rightleftharpoons B+C$, $2B\rightleftharpoons C$를 사용합니다. 반응속도는 $r_1=0.5c_A-0.05c_Bc_C$, $r_2=0.2c_B^2-0.01c_C$이고,
$$\dot c=(-r_1,\ r_1-2r_2,\ r_1+r_2)^T,\qquad y=32.84(c_A+c_B+c_C).$$
초기 실제 농도 $(0.5,0.05,0)$, 샘플 간격 0.25, 잘못된 사전 평균 $(1,0,4)$도 원본에서 가져옵니다. 원본 인쇄 RT 단위는 뒤집혀 있지만 $p=RTc$의 차원에 맞는 단위는 $\mathrm{L\,atm/mol}$입니다.

여기서는 **잡음 없는 명목 모델과 압력 정보의 구조만 재현**합니다. 원본의 $Q=10^{-6}I$, $R=0.25^2$ 잡음 실현, 모든 필터의 비교와 원본 그림 전체는 재현하지 않습니다. 같은 초기 총농도인 다른 상태 $(0,0.55,0)$는 처음 압력은 같아도 이후 압력이 달라집니다. 동역학이 추가 정보를 제공하는 이유입니다.

반응식의 보존량은 $3c_A+c_B+2c_C$입니다. 적분기 두 설정을 대조하고 이 보존량을 확인합니다. 이어 전체 명목 압력 경로에서 **초기상태만** 비음수 최소제곱으로 맞춥니다. 이는 과정 외란이 없는 초기상태 식별이며 MHE 전체 구현이나 전역 최적성의 증명이 아닙니다.

In [ ]:
def demo_book_reactor():
    from scipy.integrate import solve_ivp

    dt = 0.25
    times = np.arange(25) * dt
    initial = np.array([0.5, 0.05, 0.0])
    alternative = np.array([0.0, 0.55, 0.0])
    bad_prior = np.array([1.0, 0.0, 4.0])

    def rhs(t, x):
        ca, cb, cc = x
        r1 = 0.5 * ca - 0.05 * cb * cc
        r2 = 0.2 * cb**2 - 0.01 * cc
        return np.array([-r1, r1 - 2 * r2, r1 + r2])

    def trajectory(x0, method="DOP853", tol=1e-10):
        result = solve_ivp(
            rhs,
            (0, times[-1]),
            x0,
            t_eval=times,
            method=method,
            rtol=tol,
            atol=tol * 0.01,
        )
        if not result.success or not np.all(np.isfinite(result.y)):
            raise RuntimeError(result.message)
        return result.y.T

    truth = trajectory(initial)
    other = trajectory(alternative)
    reference = trajectory(initial, "RK45", 1e-11)
    residual = np.max(np.abs(truth - reference))
    assert residual < 1e-8
    conserved = truth @ np.array([3.0, 1.0, 2.0])
    assert np.max(np.abs(conserved - conserved[0])) < 1e-9 and truth.min() > -1e-10
    pressure = 32.84 * truth.sum(axis=1)
    other_pressure = 32.84 * other.sum(axis=1)
    assert (
        abs(pressure[0] - other_pressure[0]) < 1e-12
        and np.max(np.abs(pressure - other_pressure)) > 1
    )

    def fit_residual(x0):
        return (32.84 * trajectory(x0).sum(axis=1) - pressure) / 0.25

    fit = least_squares(
        fit_residual,
        bad_prior,
        bounds=(np.zeros(3), np.full(3, 6.0)),
        jac="3-point",
        gtol=1e-11,
        ftol=1e-11,
        xtol=1e-11,
        max_nfev=200,
    )
    assert fit.success and np.max(np.abs(fit_residual(fit.x))) < 1e-4
    assert np.linalg.norm(fit.x - initial) < 1e-4
    CHECKS["원본 반응기 적분기 대조 오차"] = float(residual)
    CHECKS["원본 반응기 보존량 오차"] = float(np.max(np.abs(conserved - conserved[0])))
    CHECKS["명목 압력 경로의 초기상태 식별 오차"] = float(
        np.linalg.norm(fit.x - initial)
    )
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for j, label in enumerate(["A", "B", "C"]):
        ax[0].plot(times, truth[:, j], label=label)
    ax[0].set(xlabel="시간", ylabel="원본 명목 농도 [mol/L]")
    ax[0].legend()
    ax[1].plot(times, pressure, label="원본 초기상태")
    ax[1].plot(times, other_pressure, "--", label="같은 초기 압력의 다른 조성")
    ax[1].set(xlabel="시간", ylabel="압력 [atm]")
    ax[1].legend(fontsize=9)
    finish_figure("book_reactor")
    print("압력 경로로 식별한 초기 농도:", fit.x)
    return truth, pressure, fit.x


demo_book_reactor()


## 4.6 On combining MHE and MPC — MHE와 MPC의 결합

**교재 인쇄 pp.318–325**

MHE 추정값을 MPC에 넣으면 추정 오차가 제어 입력과 실제 상태 변화에 영향을 줍니다. 선형 무제약 문제의 분리 원리를 제약·비선형 문제에 자동으로 확장할 수 없습니다. 교재는 입력에 대해 균일한 검출가능성, 추정 오차 경계, 강인 불변 영역과 ISS 비용 감소 조건 등을 사용합니다. 아래는 현재 측정까지 반영한 필터링형 MHE와 입력 제약 MPC를 결합하는 보충 실험입니다.

In [ ]:
def demo_mhe_mpc():
    a = 1.05
    b = 0.5
    A = np.array([[a]])
    B = np.array([[b]])
    C = np.array([[1.0]])
    Q = np.array([[0.015**2]])
    R = np.array([[0.05**2]])
    m = np.array([0.0])
    P = np.array([[1.0]])
    means = [m.copy()]
    covs = [P.copy()]
    measurements = []
    controls = []
    truth = [1.5]
    est = []
    gaps = []
    rng = np.random.default_rng(42)
    horizon = 8
    z = cp.Variable(horizon + 1)
    u = cp.Variable(horizon)
    initial = cp.Parameter()
    Pc = float(solve_discrete_are([[a]], [[b]], [[1.0]], [[0.2]]).item())
    problem = cp.Problem(
        cp.Minimize(
            cp.sum_squares(z[:-1]) + 0.2 * cp.sum_squares(u) + Pc * cp.square(z[-1])
        ),
        [z[0] == initial, z[1:] == a * z[:-1] + b * u, cp.abs(u) <= 0.8],
    )
    for k in range(35):
        y = truth[-1] + rng.normal(0, 0.05)
        measurements.append([y])
        s = max(0, k - 4)
        zz, _ = window_fit(
            A,
            C,
            Q,
            R,
            np.array(measurements[s:]),
            (means[s], covs[s]),
            B=B,
            controls=np.array(controls[s:]),
            prediction=False,
        )
        estimate = zz[-1, 0]
        L = np.linalg.solve(C @ P @ C.T + R, C @ P).T
        post = m + L @ (np.array([y]) - C @ m)
        gaps.append(abs(estimate - post[0]))
        assert gaps[-1] < 2e-6
        initial.value = estimate
        checked_solve(problem)
        uk = float(u.value[0])
        assert abs(uk) <= 0.8 + 2e-6
        est.append(estimate)
        controls.append(uk)
        I = np.eye(1) - L @ C
        Ppost = I @ P @ I.T + L @ R @ L.T
        m = A @ post + B[:, 0] * uk
        P = A @ Ppost @ A.T + Q
        means.append(m.copy())
        covs.append(P.copy())
        truth.append(a * truth[-1] + b * uk + rng.normal(0, 0.015))
    CHECKS["결합 실습 MHE–KF 필터링 최대 차이"] = float(max(gaps))
    CHECKS["결합 실습 입력 최대 크기"] = float(max(abs(np.array(controls))))
    CHECKS["결합 실습 추정 RMSE"] = float(
        np.sqrt(np.mean((np.array(est) - np.array(truth[:-1])) ** 2))
    )
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(truth, label="실제 상태")
    ax[0].plot(est, "--", label="MHE 필터링 추정")
    ax[0].set(xlabel="시간 단계 k", ylabel="상태")
    ax[0].legend()
    ax[1].step(range(len(controls)), controls, where="post")
    ax[1].axhline(0.8, color="gray", ls=":")
    ax[1].axhline(-0.8, color="gray", ls=":")
    ax[1].set(xlabel="시간 단계 k", ylabel="실제 적용 입력")
    finish_figure("mhe_mpc")


demo_mhe_mpc()


**보충 실습 — 명령 입력을 실제 적용 입력으로 착각하면 추정이 틀립니다**

입력 명령이 1.6이어도 구동기 한계가 0.8이면 실제 적용값은 0.8입니다. 추정기의 동역학에는 **실제 적용 입력**을 넣어야 합니다. 아래는 외란·측정 잡음 없는 스칼라 실험에서 같은 칼만 필터에 실제 입력 이력과 포화 전 명령 이력을 각각 넣습니다.

센서가 매번 보정하더라도 잘못된 입력 이력으로 만든 다음 상태 예측에는 오차가 남습니다. 이 비교의 포화는 구동기 동작을 모델링한 것이며 솔버 실패를 숨기려고 최적 입력을 잘라내는 처리가 아닙니다.

In [ ]:
def demo_applied_input():
    T = 24
    a = 0.9
    b = 0.5
    commands = np.r_[np.full(12, 1.6), np.full(12, -1.6)]
    applied = np.clip(commands, -0.8, 0.8)
    truth = [0.0]
    for u in applied:
        truth.append(a * truth[-1] + b * u)
    truth = np.array(truth)
    y = truth[:-1, None]
    args = (
        np.array([[a]]),
        np.ones((1, 1)),
        np.array([[0.01]]),
        np.array([[0.04]]),
        np.zeros(1),
        np.eye(1),
        y,
    )
    correct, *_ = linear_kf(*args, B=np.array([[b]]), controls=applied)
    wrong, *_ = linear_kf(*args, B=np.array([[b]]), controls=commands)
    gap = np.max(np.abs(correct[:, 0] - truth))
    bad = np.max(np.abs(wrong[:, 0] - truth))
    assert gap < 1e-12 and bad > 0.4
    CHECKS["실제 입력 이력의 무잡음 예측 오차"] = float(gap)
    CHECKS["명령 입력 이력 사용의 최대 예측 오차"] = float(bad)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].step(range(T), commands, label="포화 전 명령")
    ax[0].step(range(T), applied, label="실제 적용 입력")
    ax[0].set(xlabel="시간 단계", ylabel="입력")
    ax[0].legend()
    ax[1].plot(truth, label="실제 상태")
    ax[1].plot(correct[:, 0], "o", ms=3, label="실제 입력 사용")
    ax[1].plot(wrong[:, 0], "--", label="명령 입력 사용")
    ax[1].set(xlabel="예측 상태 시점", ylabel="상태")
    ax[1].legend()
    finish_figure("applied_input")


demo_applied_input()


## 4.7 Notes — 문헌 해설

**교재 인쇄 pp.325–327**

선형 확률 추정, 비선형 검출가능성, 제약 추정과 MHE 안정성은 서로 연결되지만 필요한 가정이 다릅니다. 문헌을 읽을 때 잡음이 가우스인지 유계인지 또는 0으로 수렴하는지, 어떤 정보 시점과 사전 비용을 사용하는지 확인합니다. 이 자료는 2017년 판의 설명을 정리하며 최신 연구 전체의 평가를 주장하지 않습니다.

## 4.8 Exercises — 연습문제

**교재 인쇄 pp.327–338**

교재 연습문제 4.1은 ISS와 수렴, 4.3은 i-IOSS와 상태 수렴, 4.4는 선형 관측가능성·검출가능성과 OSS를 다룹니다. 원본 번호로 교재를 확인하고 아래 보충 과제로 이번 실습을 확장하세요.

## 보충 과제와 힌트

1. FIE에 현재 측정 $y_T$도 포함하면 비교할 칼만 추정은 $m_T^-$인가요, $m_T^+$인가요?
2. 정확한 선형 도착 비용을 쓴 길이 1 MHE와 길이 8 MHE의 결과가 같은 이유를 설명하세요.
3. 동일한 측정을 두 번 반영하면 공분산이 작아지는 이유와 그 해석이 잘못된 이유를 설명하세요.
4. 비음수 상태 제약을 실제 음의 상태가 있는 데이터에 적용하면 어떤 문제가 생기는지 확인하세요.
5. 제곱 측정의 입자 필터에서 부호를 구분할 수 있는 추가 측정이 들어오면 사후 분포는 어떻게 달라질까요?
6. 비선형 비교에서 초기 사전과 창 길이를 바꾸고 RMSE·실행 시간을 비교하세요. 한 방법이 항상 가장 좋은가요?

<details><summary>힌트</summary>

1. 현재 측정까지 쓰므로 $m_T^+$입니다. 기본 실습은 $y_{0:T-1}$만 사용합니다.
2. 창 밖 정보가 정확한 도착 비용에 이미 들어 있습니다. 비선형 이차 근사에서는 이 일치가 일반적으로 사라집니다.
3. 독립적이지 않은 정보를 독립인 새 측정처럼 취급하여 과도하게 확신하게 됩니다.
4. 추정 제약은 실제 공정을 바꾸지 못합니다. 편향 또는 실행 불가능성을 만들 수 있습니다.
5. 추가 센서가 부호를 알려 주면 한 봉우리의 가중치가 커질 수 있습니다.
6. 같은 데이터·사전·정보 시점을 유지한 비교와 여러 난수 시드의 비교를 구분하세요.

</details>

## 실행 검증 결과

기본 설정의 수치 일치와 제약·공분산·확률 가중치 검증을 기록합니다. RMSE 순위와 계산 시간은 이 실험의 결과입니다.

In [ ]:
for name, value in CHECKS.items():
    print(f"{name}: {value:.8g}")
print("검증 지표 수:", len(CHECKS))
print(
    {
        "Python": sys.version.split()[0],
        "NumPy": np.__version__,
        "SciPy": scipy.__version__,
        "Matplotlib": matplotlib.__version__,
        "CVXPY": cp.__version__,
    }
)


## 참고 자료

- Rawlings, Mayne, Diehl (2017), *Model Predictive Control: Theory, Computation, and Design*, 2nd edition, 1st printing, Chapter 4, pp.269–338.
- 예측형 정보 구간은 교재 p.271 각주에 따릅니다. §4.3.4와 §4.6의 실습은 평활화·필터링형으로 별도 표시했습니다.
- Example 4.40은 원본 식의 수치 적용, Example 4.41은 명목 모델·측정 구조의 재현이며 나머지는 보충 실습입니다. 입자 필터와 구간 집합 추정은 교재의 짧은 논의를 확장한 별도 예입니다.
- 비선형 MHE는 근사 도착 비용의 국소 최소제곱 해이며 정확한 사후 평균·전역 최적해를 주장하지 않습니다.
- 한글 글꼴은 Google Fonts의 나눔고딕을 사용합니다.